# Getting Started with Ollama: Chat Models and Embeddings



## 1. Install or update Ollama

Install/update the server from [Ollama Downloads](https://ollama.com/download).
Updating the Python package alone does not update the server. Open the Ollama desktop
app, or run `ollama serve` in a terminal if no server is already running.

Useful terminal commands:

```bash
ollama --version
ollama list
ollama ps
```

Download at least one chat model and one embedding model. To run all default comparisons,
pull the following models (each download uses disk space and bandwidth):

```bash
ollama pull llama3.2:1b
ollama pull gemma3:1b
ollama pull qwen3:0.6b
ollama pull all-minilm
ollama pull embeddinggemma
```

Chat candidates are small examples, not a list of the newest or best models. You can use
any suitable local model from the [model library](https://ollama.com/library). For example,
replace a candidate with a model already installed on your computer. Dedicated embedding
models are listed separately: [all-minilm](https://ollama.com/library/all-minilm),
[EmbeddingGemma](https://ollama.com/library/embeddinggemma).


In [ ]:
%pip install -e ".[ollama]"

## 2. General configuration: edit .env

- Choose exact model tags in `.env` using `ollama list` or the library. Native and Docker
- Ollama both use the configured host URL; see `README.md` for the matching pull commands. The selected `OLLAMA_MODEL` and `OLLAMA_EMBED_MODEL` are used for single-model examples. The comparison cells iterate over the whole lists.
- Missing models are reported and skipped. Set `DOWNLOAD_MISSING = True` only when you want the notebook to download the selected models automatically.

- Keep chat and embedding models separate. Do not assume a chat model provides useful embeddings. Cloud model tags require a different setup; this notebook uses local models.

In [ ]:
from notebook_utils import load_lab_environment
ENV = load_lab_environment()
BASE_URL = ENV.get("OLLAMA_BASE_URL", "http://localhost:11434")
CHAT_MODEL = ENV.get("OLLAMA_MODEL", "qwen3:0.6b")
EMBED_MODEL = ENV.get("OLLAMA_EMBED_MODEL", "all-minilm")
CHAT_MODELS = list(dict.fromkeys([CHAT_MODEL] + [x.strip() for x in ENV.get("OLLAMA_CHAT_MODELS", "").split(",") if x.strip()]))
EMBED_MODELS = list(dict.fromkeys([EMBED_MODEL] + [x.strip() for x in ENV.get("OLLAMA_EMBED_MODELS", "").split(",") if x.strip()]))
DOWNLOAD_MISSING = False

TEMPERATURE = float(ENV.get("TEMPERATURE", "0.2"))
SEED = 42
NUM_CTX = int(ENV.get("OLLAMA_NUM_CTX", "4096"))
MAX_NEW_TOKENS = int(ENV.get("MAX_OUTPUT_TOKENS", "512"))
REQUEST_TIMEOUT = int(ENV.get("REQUEST_TIMEOUT_SECONDS", "180"))
KEEP_ALIVE = "0"
SYSTEM_PROMPT = "You are a concise teaching assistant. Use clear, accurate explanations."
TEST_PROMPT = "Explain retrieval-augmented generation in exactly three short bullet points."

## 3. Connect, print versions, and discover installed models

The server and Python package have separate version numbers. These are recorded for reproducibility. `/api/version` and `/api/tags` describe the server currently running at `BASE_URL`. Connection problems stop this setup cell with an actionable message.

In [ ]:
import json
import time
from functools import lru_cache
from importlib.metadata import version

import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown
from ollama import Client, ResponseError

client = Client(host=BASE_URL, timeout=REQUEST_TIMEOUT)
try:
    server_response = requests.get(f"{BASE_URL}/api/version", timeout=10)
    server_response.raise_for_status()
    SERVER_VERSION = server_response.json()["version"]
    local_models = client.list().models
except Exception as exc:
    raise RuntimeError(
        f"Cannot reach Ollama at {BASE_URL}. Open the Ollama app or run 'ollama serve', "
        "then rerun this cell. If using Colab, switch to a local notebook kernel."
    ) from exc

CLIENT_VERSION = version("ollama")
print("Ollama server:", SERVER_VERSION)
print("Python client:", CLIENT_VERSION)
print("Endpoint:", BASE_URL)

model_rows = []
for item in local_models:
    details = item.details
    model_rows.append({
        "model": item.model,
        "size_GB_on_disk": round((item.size or 0) / 1e9, 2),
        "parameters": details.parameter_size if details else None,
        "quantization": details.quantization_level if details else None,
    })
display(pd.DataFrame(model_rows, columns=["model", "size_GB_on_disk", "parameters", "quantization"]))
print("These are downloaded sizes, not estimates of runtime RAM/VRAM requirements.")

### Check capabilities and optionally download missing models

`client.show(...)` exposes capabilities such as `completion`, `embedding`, and `thinking`.
The helpers below validate the intended use and avoid sending chat requests to an embedding
model. Model aliases without a tag are matched to `:latest`; use explicit tags for class work.

For thinking-capable chat models, the helper sends `think=False` to make this short comparison
focus on final answers. Other models receive no thinking parameter. Model families can impose
additional constraints; an API error is recorded rather than silently replacing a model.

In [ ]:
def canonical_name(name):
    return name if ":" in name else name + ":latest"


@lru_cache(maxsize=None)
def model_info(name):
    return client.show(name)


def prepare_model(name, capability):
    installed = {item.model for item in client.list().models}
    if canonical_name(name) not in installed:
        if not DOWNLOAD_MISSING:
            raise ValueError(f"Model {name!r} is missing. Run: ollama pull {name}")
        print(f"Downloading {name}...")
        for progress in client.pull(name, stream=True):
            if progress.completed is not None and progress.total:
                print(f"\r{progress.status}: {100 * progress.completed / progress.total:.0f}%", end="", flush=True)
        print()
        model_info.cache_clear()
    info = model_info(name)
    capabilities = info.capabilities or []
    if capability not in capabilities:
        raise ValueError(f"{name} does not advertise {capability!r}; capabilities: {capabilities}")
    return info


def chat_kwargs(name):
    info = prepare_model(name, "completion")
    settings = {
        "model": name,
        "options": {"temperature": TEMPERATURE, "seed": SEED,
                    "num_ctx": NUM_CTX, "num_predict": MAX_NEW_TOKENS},
        "keep_alive": KEEP_ALIVE,
    }
    if "thinking" in (info.capabilities or []):
        settings["think"] = False
    return settings


capability_rows = []
for model, capability in [(m, "completion") for m in CHAT_MODELS] + [(m, "embedding") for m in EMBED_MODELS]:
    try:
        info = prepare_model(model, capability)
        capability_rows.append({"model": model, "capabilities": ", ".join(info.capabilities or []), "status": "ready"})
    except Exception as exc:
        capability_rows.append({"model": model, "capabilities": "", "status": str(exc)})
display(pd.DataFrame(capability_rows))

## 4. Try a single chat model

Change `CHAT_MODEL` in the configuration cell to try another model. Requests use the
same system instruction and prompt, with explicit non-streaming output. The official
client returns typed response objects, so the answer is `response.message.content`.

In [ ]:
single_chat_response = None
try:
    single_chat_response = client.chat(
        **chat_kwargs(CHAT_MODEL),
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": TEST_PROMPT}],
        stream=False,
    )
    display(Markdown(single_chat_response.message.content or "(No final answer text returned.)"))
    print("Finish reason:", single_chat_response.done_reason)
except Exception as exc:
    print("Chat failed:", exc)

## 5. Compare several chat models on the same task

For each model, record the output, elapsed time, model load time and generation throughput.
Requests run sequentially. With `KEEP_ALIVE = "0"`, each model is unloaded after the request;
loading therefore contributes to end-to-end latency. The first run may be slower.

Temperature and seed make conditions easier to reproduce, but outputs are not guaranteed
identical across models, hardware or server versions. Tokenizers differ, so tokens/second is
not a direct quality comparison. Manually assess correctness, clarity and format compliance.

In [ ]:
def response_metrics(response, elapsed):
    eval_seconds = (response.eval_duration or 0) / 1e9
    return {
        "elapsed_s": round(elapsed, 3),
        "server_total_s": round((response.total_duration or 0) / 1e9, 3),
        "load_s": round((response.load_duration or 0) / 1e9, 3),
        "generated_tokens": response.eval_count,
        "generation_tokens_per_s": round((response.eval_count or 0) / eval_seconds, 2) if eval_seconds else None,
        "finish_reason": response.done_reason,
    }


chat_results = []
for model in CHAT_MODELS:
    record = {"model": model, "prompt": TEST_PROMPT}
    try:
        settings = chat_kwargs(model)  # preparation time is outside the timed generation
        started = time.perf_counter()
        result = client.chat(
            **settings,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": TEST_PROMPT}],
            stream=False,
        )
        record.update(response_metrics(result, time.perf_counter() - started))
        record.update(status="ok", answer=result.message.content or "")
    except Exception as exc:
        record.update(status="failed", error=str(exc), answer="")
    chat_results.append(record)

chat_comparison = pd.DataFrame(chat_results)
display(chat_comparison.drop(columns=["answer", "prompt"]))
for record in chat_results:
    if record["status"] == "ok":
        display(Markdown(f"### {record['model']}"))
        display(Markdown(record["answer"] or "(No final answer text.)"))

### Streaming a response

Streaming emits partial message content instead of waiting for the full answer. Empty
content chunks are normal. We display final-answer content, not a model's thinking trace.

In [ ]:
try:
    stream = client.chat(
        **chat_kwargs(CHAT_MODEL),
        messages=[{"role": "user", "content": "Give two practical uses of RAG."}],
        stream=True,
    )
    for part in stream:
        print(part.message.content or "", end="", flush=True)
    print()
except Exception as exc:
    print("Streaming failed:", exc)

## 6. Generate an embedding with Ollama

An embedding model produces a numeric vector rather than a conversational answer. Use
**`client.embed(model=..., input=...)`**, with `response.embeddings[0]` for a single input.
This replaces the older `client.embeddings(..., prompt=...)` interface.

`truncate=False` makes overlong input an error instead of silently truncating it. Chunk long
documents before embedding them. Embedding dimensions depend on the model; never compare
vectors from different models, even if their dimensions happen to match.

In [ ]:
def embed_texts(model, texts):
    prepare_model(model, "embedding")
    response = client.embed(model=model, input=texts, truncate=False, keep_alive=KEEP_ALIVE)
    vectors = np.asarray(response.embeddings, dtype=float)
    if vectors.ndim != 2 or vectors.shape[0] != len(texts) or not np.isfinite(vectors).all():
        raise ValueError("Unexpected embedding shape or non-finite vector values")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("The model returned a zero embedding")
    return vectors / norms  # explicit normalization makes cosine calculations clear


single_vector = None
try:
    single_vector = embed_texts(EMBED_MODEL, ["RAG retrieves documents before generating an answer."])[0]
    print("Model:", EMBED_MODEL)
    print("Vector dimensions:", single_vector.shape[0])
    print("First eight values:", single_vector[:8])
    print("L2 norm:", np.linalg.norm(single_vector))
except Exception as exc:
    print("Embedding failed:", exc)

## 7. Batch embeddings and semantic similarity

A list passed to `input` returns one vector per text, in the same order. Compare a pair of
paraphrases with unrelated sentences. Similarity is a model-dependent score, not a probability.
Ollama returns normalized embeddings; we normalize explicitly as a check and for teaching.

In [ ]:
SENTENCES = [
    "A cat is sleeping on the sofa.",
    "A kitten is resting on the couch.",
    "RAG uses retrieved documents to help answer questions.",
    "The weather forecast predicts heavy rain tomorrow.",
]
try:
    sentence_vectors = embed_texts(EMBED_MODEL, SENTENCES)
    print("Batch shape (texts, dimensions):", sentence_vectors.shape)
    similarities = sentence_vectors @ sentence_vectors.T
    display(pd.DataFrame(similarities, index=[f"S{i}" for i in range(len(SENTENCES))],
                         columns=[f"S{i}" for i in range(len(SENTENCES))]).round(3))
    for i, sentence in enumerate(SENTENCES):
        print(f"S{i}: {sentence}")
except Exception as exc:
    print("Batch embedding failed:", exc)

## 8. Compare embedding models on semantic retrieval

Each model embeds the same documents and questions **in its own vector space**. We compare
rankings and a small labeled hit@1 score, not vector coordinates or raw score magnitudes across
models. Changing the embedding model requires re-embedding the whole collection.

Some embedding models prescribe different prefixes for documents and queries. The helper
below applies EmbeddingGemma's search prefixes; all-minilm uses plain text. If you add another
model, check its model card and adapt this helper. These tiny examples are a demonstration,
not a reliable benchmark of general retrieval quality.

In [ ]:
DOCUMENTS = [
    {"id": "rag", "text": "Retrieval-augmented generation retrieves relevant documents before generating an answer."},
    {"id": "vae", "text": "A variational autoencoder learns a latent distribution and reconstructs data using a decoder."},
    {"id": "diffusion", "text": "Diffusion models generate samples by repeatedly denoising a noisy input."},
    {"id": "embedding", "text": "Text embeddings represent meaning as vectors for semantic similarity and search."},
]
SEARCH_CASES = [
    {"question": "How can an assistant use documents when answering?", "expected_id": "rag"},
    {"question": "Which method generates samples through iterative denoising?", "expected_id": "diffusion"},
    {"question": "How can I represent sentence meaning numerically?", "expected_id": "embedding"},
]


def retrieval_inputs(model, texts, *, query):
    if model.split(":")[0] == "embeddinggemma":
        prefix = "task: search result | query: " if query else "title: none | text: "
        return [prefix + text for text in texts]
    return texts


retrieval_cache = {}
embedding_results = []
retrieval_rows = []
for model in EMBED_MODELS:
    try:
        prepare_model(model, "embedding")
        started = time.perf_counter()
        document_vectors = embed_texts(model, retrieval_inputs(model, [d["text"] for d in DOCUMENTS], query=False))
        query_vectors = embed_texts(model, retrieval_inputs(model, [c["question"] for c in SEARCH_CASES], query=True))
        scores = query_vectors @ document_vectors.T
        correct = 0
        for case, row in zip(SEARCH_CASES, scores):
            order = np.argsort(-row)
            top_id = DOCUMENTS[int(order[0])]["id"]
            correct += top_id == case["expected_id"]
            retrieval_rows.append({"model": model, "question": case["question"],
                                   "expected_id": case["expected_id"], "top_id": top_id,
                                   "top_score": round(float(row[order[0]]), 4),
                                   "ranking": [DOCUMENTS[int(i)]["id"] for i in order]})
        retrieval_cache[model] = document_vectors
        embedding_results.append({"model": model, "status": "ok", "dimensions": document_vectors.shape[1],
                                  "hit_at_1": correct / len(SEARCH_CASES),
                                  "elapsed_s": round(time.perf_counter() - started, 3)})
    except Exception as exc:
        embedding_results.append({"model": model, "status": "failed", "error": str(exc)})
display(pd.DataFrame(embedding_results))
display(pd.DataFrame(retrieval_rows))